# HIRO PyTorch on Google Colab

This notebook installs dependencies, clones the repository, runs basic smoke tests, and provides short training/evaluation experiments for both **HIRO** and **TD3**.

## Notes
- Original repository: `watakandai/hiro_pytorch`
- The original code depends on **MuJoCo** through `mujoco_py` and legacy `gym` APIs.
- Colab compatibility for old MuJoCo stacks can be brittle; this notebook pins versions and patches execution where useful.
- For fast validation, use the short experiments defined below rather than the repository defaults.

## What this notebook does

1. Install system packages
2. Install Python dependencies
3. Clone the repository
4. Apply a small notebook-only runtime helper
5. Run smoke tests
6. Run short HIRO / TD3 experiments
7. Evaluate trained checkpoints

In [ ]:
# ===== 1) System dependencies =====
# Colab already has many basics, but install the common libraries used by mujoco_py.

!apt-get -y update
!apt-get install -y \
    libosmesa6-dev \
    libgl1-mesa-glx \
    libglfw3 \
    patchelf \
    xvfb \
    swig \
    cmake \
    git

print('System packages installed.')

In [ ]:
# ===== 2) Python dependencies =====
# Pin older versions to better match mujoco_py-era code.

!python -m pip install --upgrade pip setuptools wheel
!python -m pip install numpy==1.23.5
!python -m pip install torch
!python -m pip install gym==0.17.3
!python -m pip install mujoco-py==2.1.2.14
!python -m pip install matplotlib pandas pillow imageio imageio-ffmpeg tqdm

print('Python packages installed.')

## MuJoCo note

If MuJoCo import/build fails in later cells, the failure is usually due to binary or OpenGL compatibility in Colab rather than the repository code itself.

The notebook still gives you:
- reproducible installation commands,
- repository setup,
- smoke-test helpers,
- runnable experiment cells for environments that successfully initialize.

In [ ]:
# ===== 3) Clone repository =====

import os

REPO_URL = 'https://github.com/watakandai/hiro_pytorch.git'
REPO_DIR = '/content/hiro_pytorch'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repository already exists, skipping clone.')

%cd /content/hiro_pytorch
!git rev-parse HEAD

In [ ]:
# ===== 4) Environment variables for headless rendering / mujoco =====

import os

os.environ['MUJOCO_GL'] = 'osmesa'
os.environ['PYOPENGL_PLATFORM'] = 'osmesa'
os.environ['LD_LIBRARY_PATH'] = '/usr/lib/x86_64-linux-gnu:' + os.environ.get('LD_LIBRARY_PATH', '')

print('MUJOCO_GL =', os.environ['MUJOCO_GL'])
print('PYOPENGL_PLATFORM =', os.environ['PYOPENGL_PLATFORM'])

In [ ]:
# ===== 5) Basic repository inspection =====

!ls -la
!find envs -maxdepth 2 -type f | sort | sed -n '1,200p'
!find hiro -maxdepth 2 -type f | sort | sed -n '1,200p'

In [ ]:
# ===== 6) Notebook runtime helper =====
# This helper runs experiments from notebook cells without requiring CLI invocation.

from pathlib import Path

helper_code = r'''
import os
import copy
import datetime
from types import SimpleNamespace

import numpy as np

from envs import EnvWithGoal
from envs.create_maze_env import create_maze_env
from hiro.models import HiroAgent, TD3Agent
from hiro.utils import Logger, _is_update, record_experience_to_csv, listdirs


def build_args(**overrides):
    defaults = dict(
        train=False,
        eval=False,
        render=False,
        save_video=False,
        sleep=-1,
        eval_episodes=5,
        env='AntMaze',
        td3=False,
        num_episode=10,
        start_training_steps=100,
        writer_freq=25,
        subgoal_dim=15,
        load_episode=-1,
        model_save_freq=50,
        print_freq=5,
        exp_name=None,
        model_path='model',
        log_path='log',
        policy_freq_low=2,
        policy_freq_high=2,
        buffer_size=50000,
        batch_size=64,
        buffer_freq=10,
        train_freq=10,
        reward_scaling=0.1,
    )
    defaults.update(overrides)
    return SimpleNamespace(**defaults)


def make_experiment_name(prefix='colab'):
    return prefix + '_' + datetime.datetime.now().strftime('%Y%m%d_%H%M%S')


def build_env_and_agent(args):
    env = EnvWithGoal(create_maze_env(args.env), args.env)
    goal_dim = 2
    state_dim = env.state_dim
    action_dim = env.action_dim
    scale = env.action_space.high * np.ones(action_dim)

    exp_name = args.exp_name or make_experiment_name('td3' if args.td3 else 'hiro')

    if args.td3:
        agent = TD3Agent(
            state_dim=state_dim,
            action_dim=action_dim,
            goal_dim=goal_dim,
            scale=scale,
            model_save_freq=args.model_save_freq,
            model_path=os.path.join(args.model_path, exp_name),
            buffer_size=args.buffer_size,
            batch_size=args.batch_size,
            start_training_steps=args.start_training_steps,
        )
    else:
        agent = HiroAgent(
            state_dim=state_dim,
            action_dim=action_dim,
            goal_dim=goal_dim,
            subgoal_dim=args.subgoal_dim,
            scale_low=scale,
            start_training_steps=args.start_training_steps,
            model_path=os.path.join(args.model_path, exp_name),
            model_save_freq=args.model_save_freq,
            buffer_size=args.buffer_size,
            batch_size=args.batch_size,
            buffer_freq=args.buffer_freq,
            train_freq=args.train_freq,
            reward_scaling=args.reward_scaling,
            policy_freq_high=args.policy_freq_high,
            policy_freq_low=args.policy_freq_low,
        )

    return env, agent, exp_name


class NotebookTrainer:
    def __init__(self, args, env, agent, experiment_name):
        self.args = args
        self.env = env
        self.agent = agent
        log_path = os.path.join(args.log_path, experiment_name)
        self.logger = Logger(log_path=log_path)

    def log(self, global_step, data):
        losses, td_errors = data
        if global_step >= self.args.start_training_steps and _is_update(global_step, self.args.writer_freq):
            for k, v in losses.items():
                self.logger.write('loss/%s' % k, v, global_step)
            for k, v in td_errors.items():
                self.logger.write('td_error/%s' % k, v, global_step)

    def evaluate(self, e):
        if _is_update(e, self.args.print_freq):
            agent = copy.deepcopy(self.agent)
            rewards, success_rate = agent.evaluate_policy(self.env)
            self.logger.write('Success Rate', success_rate, e)
            print(
                'episode:{:05d}, mean:{:.2f}, std:{:.2f}, median:{:.2f}, success:{:.2f}'.format(
                    e, np.mean(rewards), np.std(rewards), np.median(rewards), success_rate
                )
            )

    def train(self):
        global_step = 0
        for e in np.arange(self.args.num_episode) + 1:
            obs = self.env.reset()
            fg = obs['desired_goal']
            s = obs['observation']
            done = False
            step = 0
            episode_reward = 0

            self.agent.set_final_goal(fg)

            while not done:
                a, r, n_s, done = self.agent.step(s, self.env, step, global_step, explore=True)
                self.agent.append(step, s, a, n_s, r, done)
                losses, td_errors = self.agent.train(global_step)
                self.log(global_step, (losses, td_errors))

                s = n_s
                episode_reward += r
                step += 1
                global_step += 1
                self.agent.end_step()

            self.agent.end_episode(e, self.logger)
            self.logger.write('reward/Reward', episode_reward, e)
            self.evaluate(e)


def run_training_experiment(**kwargs):
    args = build_args(train=True, **kwargs)
    env, agent, exp_name = build_env_and_agent(args)
    record_experience_to_csv(args, exp_name)
    trainer = NotebookTrainer(args, env, agent, exp_name)
    trainer.train()
    return dict(experiment_name=exp_name, model_dir=os.path.join(args.model_path, exp_name), log_dir=os.path.join(args.log_path, exp_name))


def run_evaluation_experiment(exp_name, **kwargs):
    args = build_args(eval=True, exp_name=exp_name, **kwargs)
    env, agent, _ = build_env_and_agent(args)
    agent.load(args.load_episode)
    rewards, success_rate = agent.evaluate_policy(env, args.eval_episodes, args.render, args.save_video, args.sleep)
    print('mean:{:.2f}, std:{:.2f}, median:{:.2f}, success:{:.2f}'.format(np.mean(rewards), np.std(rewards), np.median(rewards), success_rate))
    return rewards, success_rate
'''

Path('colab_runtime.py').write_text(helper_code)
print('Wrote colab_runtime.py')

In [ ]:
# ===== 7) Import smoke test =====

import importlib
import traceback

modules = [
    'envs',
    'envs.create_maze_env',
    'hiro.hiro_utils',
    'hiro.models',
    'hiro.utils',
    'colab_runtime',
]

for m in modules:
    try:
        importlib.import_module(m)
        print(f'OK: {m}')
    except Exception as e:
        print(f'FAILED: {m}')
        traceback.print_exc()

In [ ]:
# ===== 8) Environment construction smoke test =====

import traceback

try:
    from envs import EnvWithGoal
    from envs.create_maze_env import create_maze_env

    env = EnvWithGoal(create_maze_env('AntMaze'), 'AntMaze')
    obs = env.reset()
    print('Environment created successfully.')
    print('Observation keys:', obs.keys())
    print('Observation shape:', obs['observation'].shape)
    print('Desired goal:', obs['desired_goal'])
    print('Action dim:', env.action_dim)
    print('State dim:', env.state_dim)
except Exception as e:
    print('Environment creation failed.')
    traceback.print_exc()

## Short experiments

The repository defaults are very large (`25000` episodes), so for Colab we use much smaller runs for smoke testing and demonstration.

In [ ]:
# ===== 9) Short HIRO training run =====

from colab_runtime import run_training_experiment

hiro_result = run_training_experiment(
    td3=False,
    env='AntMaze',
    num_episode=3,
    start_training_steps=10,
    writer_freq=5,
    print_freq=1,
    model_save_freq=2,
    batch_size=32,
    buffer_size=5000,
    buffer_freq=10,
    train_freq=10,
    reward_scaling=0.1,
    exp_name='colab_hiro_demo'
)

print(hiro_result)

In [ ]:
# ===== 10) Short TD3 training run =====

from colab_runtime import run_training_experiment

td3_result = run_training_experiment(
    td3=True,
    env='AntMaze',
    num_episode=3,
    start_training_steps=10,
    writer_freq=5,
    print_freq=1,
    model_save_freq=2,
    batch_size=32,
    buffer_size=5000,
    exp_name='colab_td3_demo'
)

print(td3_result)

In [ ]:
# ===== 11) Evaluate HIRO checkpoint =====

from colab_runtime import run_evaluation_experiment

try:
    hiro_rewards, hiro_success = run_evaluation_experiment(
        'colab_hiro_demo',
        td3=False,
        env='AntMaze',
        eval_episodes=2,
        load_episode=-1,
        render=False,
        save_video=False,
    )
except Exception as e:
    import traceback
    traceback.print_exc()

In [ ]:
# ===== 12) Evaluate TD3 checkpoint =====

from colab_runtime import run_evaluation_experiment

try:
    td3_rewards, td3_success = run_evaluation_experiment(
        'colab_td3_demo',
        td3=True,
        env='AntMaze',
        eval_episodes=2,
        load_episode=-1,
        render=False,
        save_video=False,
    )
except Exception as e:
    import traceback
    traceback.print_exc()

In [ ]:
# ===== 13) Show generated artifacts =====

!find model -maxdepth 3 -type f | sort | sed -n '1,200p'
!find log -maxdepth 3 -type f | sort | sed -n '1,200p'

## Customization

You can increase training length by changing:
- `num_episode`
- `buffer_size`
- `batch_size`
- `start_training_steps`
- `model_save_freq`

Example:
```python
longer_run = run_training_experiment(
    td3=False,
    env='AntMaze',
    num_episode=20,
    start_training_steps=100,
    batch_size=64,
    buffer_size=20000,
    exp_name='colab_hiro_longer'
)
```

## Troubleshooting

### If `mujoco_py` fails to build/import
- Restart the runtime after installation and rerun from the top.
- Ensure the runtime is using Python 3 and Linux Colab.
- Re-run the system dependency cell, then the Python install cell.

### If environment creation fails
- The most likely cause is MuJoCo / OpenGL compatibility in Colab.
- The repo code itself expects legacy Gym MuJoCo APIs.

### If training is too slow
- Lower `num_episode`
- Lower `batch_size`
- Use shorter smoke tests first

### If you only want import-level verification
- Run cells through the import smoke test and skip training.